In [1]:
import pandas as pd, matplotlib.pyplot as plt, numpy as np, scipy
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType, StringType, ArrayType, MapType
from pyspark.sql.functions import col, array, struct, lit, udf, pandas_udf, broadcast

spark = SparkSession.builder.appName("MySparkApp").getOrCreate()

# **Spark Session**

- Initialize the spark session
  - **`.builder()`** sets up a session
  - **`.appName()`** manages multiple session
  - **`.getOrCreate()`** creates or retrives a session

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MySparkApp").getOrCreate()
print(spark)

# **PySpark DataFrames**

In [7]:
df = spark.read.csv("salaries.csv", header=True, inferSchema=True)
df.show(5)

+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|           job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|     2020|              EN|             FT| Azure Data Engineer|100000|            USD|       100000|                MU|           0|              MU|           S|
|     2020|              EN|             CT|  Staff Data Analyst| 60000|            CAD|        44753|                CA|          50|              CA|           L|
|     2020|              SE|             FT|Staff Data Scientist|164000|            USD|       164000|                US|          50|              US|           M|
|     2020

# **Printing DataFrame Schema**

In [9]:
df.printSchema()

root
 |-- work_year: integer (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- remote_ratio: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)



## **Row count**

In [10]:
df.count()

37234

## **GroupBy**

- group rows based on one or more columns

### **`.agg()`**

- applies aggregate functions to grouped data

In [19]:
df.groupBy("experience_level").agg({"salary_in_usd": "avg"}).show() #.sum, .min, .max, etc.

+----------------+------------------+
|experience_level|avg(salary_in_usd)|
+----------------+------------------+
|              EX|198208.34306569342|
|              MI|144187.63228574092|
|              EN|107310.08243840809|
|              SE|174433.86120854237|
+----------------+------------------+



## **Select**

- select specific ***columns*** from the DataFrame

In [21]:
df.filter(df['experience_level']=='EX').show(5)

+---------+----------------+---------------+------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|         job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|     2020|              EX|             FT|    Data Scientist|300000|            USD|       300000|                US|         100|              US|           L|
|     2020|              EX|             FT|Staff Data Analyst| 15000|            USD|        15000|                NG|           0|              CA|           M|
|     2020|              EX|             FT|     Data Engineer| 70000|            EUR|        79833|                ES|          50|              ES|           L|
|     2020|           

## **Filter or Where**

- filter **`rows`** based on specific conditions

In [22]:
df.select('experience_level', 'salary').show(5)

+----------------+------+
|experience_level|salary|
+----------------+------+
|              EN|100000|
|              EN| 60000|
|              SE|164000|
|              EN| 42000|
|              EX|300000|
+----------------+------+
only showing top 5 rows


ทำทั้งคู่

In [23]:
df.filter(df['experience_level']=='EX').select('experience_level', 'salary').show(5)

+----------------+------+
|experience_level|salary|
+----------------+------+
|              EX|300000|
|              EX| 15000|
|              EX| 70000|
|              EX|325000|
|              EX|250000|
+----------------+------+
only showing top 5 rows


- **`groupBy()`** แบบไม่มี argument หมายถึง "aggregate ทั้ง DataFrame เป็นกลุ่มเดียว"

In [37]:
df.filter(df["company_location"]=="CA").filter(df['experience_level']=="EN").groupBy().avg("salary_in_usd").show()

+------------------+
|avg(salary_in_usd)|
+------------------+
| 97330.53932584269|
+------------------+



มีผลแบบเดียวกับ pandas df แบบนี้

In [34]:
pdf = pd.read_csv('/content/salaries.csv')

pdf = pdf[(pdf["company_location"]=="CA") & (pdf['experience_level']=="EN")]["salary_in_usd"]
pdf.mean()

np.float64(97330.53932584269)

# **`.sort()` หรือ `.orderBy()`**


In [43]:
df.select('experience_level', 'salary').sort("salary").show(5)

+----------------+------+
|experience_level|salary|
+----------------+------+
|              EN| 14000|
|              EN| 14400|
|              EN| 14400|
|              EN| 14400|
|              EN| 15000|
+----------------+------+
only showing top 5 rows


In [45]:
df.select('experience_level', 'salary').orderBy("salary", ascending=False).show(5)

+----------------+--------+
|experience_level|  salary|
+----------------+--------+
|              MI|30400000|
|              MI|11000000|
|              MI|11000000|
|              MI| 8500000|
|              SE| 7500000|
+----------------+--------+
only showing top 5 rows


# **Drop missing values**

In [46]:
df.na.drop().show(5)

+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|           job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|     2020|              EN|             FT| Azure Data Engineer|100000|            USD|       100000|                MU|           0|              MU|           S|
|     2020|              EN|             CT|  Staff Data Analyst| 60000|            CAD|        44753|                CA|          50|              CA|           L|
|     2020|              SE|             FT|Staff Data Scientist|164000|            USD|       164000|                US|          50|              US|           M|
|     2020

# **Creating dataframe from json**

In [41]:
jdf = spark.read.json("/content/adults.json")
jdf.show(5)

+---+-------------+------+--------------+-----------------+
|age|education.num|income|marital.status|       occupation|
+---+-------------+------+--------------+-----------------+
| 90|            9| <=50K|       Widowed|                ?|
| 82|            9| <=50K|       Widowed|  Exec-managerial|
| 66|           10| <=50K|       Widowed|                ?|
| 54|            4| <=50K|      Divorced|Machine-op-inspct|
| 41|           10| <=50K|     Separated|   Prof-specialty|
+---+-------------+------+--------------+-----------------+
only showing top 5 rows


In [48]:
jdf.filter(jdf["age"]>40).show(5)

+---+-------------+------+--------------+-----------------+
|age|education.num|income|marital.status|       occupation|
+---+-------------+------+--------------+-----------------+
| 90|            9| <=50K|       Widowed|                ?|
| 82|            9| <=50K|       Widowed|  Exec-managerial|
| 66|           10| <=50K|       Widowed|                ?|
| 54|            4| <=50K|      Divorced|Machine-op-inspct|
| 41|           10| <=50K|     Separated|   Prof-specialty|
+---+-------------+------+--------------+-----------------+
only showing top 5 rows


# **DataTypes Syntax for PySpark DataFrames**

In [42]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType

# Construct the schema
schema = StructType([StructField("id", IntegerType(), True),
                     StructField("name", StringType(), True),
                     StructField("scores", ArrayType(IntegerType()), True)])
# อย่ารัน #
df = spark.createDataFrame(data, schema=schema)

In [50]:
jdf.printSchema()

root
 |-- age: long (nullable = true)
 |-- education.num: long (nullable = true)
 |-- income: string (nullable = true)
 |-- marital.status: string (nullable = true)
 |-- occupation: string (nullable = true)



In [65]:
json_df = pd.read_json("/content/adults.json", lines=True)
json_df = json_df.head(100)
json_df.to_csv('adult_reduced_100.csv')
pdf = pd.read_csv('/content/adult_reduced_100.csv').drop(columns='Unnamed: 0')
pdf.to_csv('adult_reduced_100.csv')


In [72]:
schema = StructType([StructField("age",IntegerType()),
                     StructField("education_num", IntegerType()),
                     StructField("marital_status",StringType()),
                     StructField("occupation", StringType()),
                     StructField("income", StringType()),
                    ])
census_adult = spark.read.csv("/content/adult_reduced_100.csv", header=True)#, schema=schema)
census_adult.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- age: string (nullable = true)
 |-- education.num: string (nullable = true)
 |-- marital.status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- income: string (nullable = true)



จะเห็นว่า education กับ age เป็น string
- เปลี่ยนเป็น integer ได้ดังนี้

In [70]:
census_adult = spark.read.csv("/content/adult_reduced_100.csv", header=False, schema=schema)
census_adult.printSchema()

root
 |-- age: integer (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- income: string (nullable = true)



# **Data Manipulation with DataFrames**

- **`df.na.drop()`** : remove *rows* with missing values
- **`df.na.fill()`** : replace missing values with default values
- **`df.withColumn()`** : add or transform columns
- **`df.withColumnRenamed()`** : rename column
- **`df.drop()`** : drop columns
- **`df.filter()`**, **`df.groupBy()`**

In [32]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MySparkApp").getOrCreate()
schema = StructType([StructField("age",IntegerType()),
                     StructField("education.num", IntegerType()),
                     StructField("marital.status",StringType()),
                     StructField("occupation", StringType()),
                     StructField("income", StringType()),
                    ])
df = spark.read.json("adults.json", schema=schema)
print(df.show(5))
df.count()

+---+-------------+--------------+-----------------+------+
|age|education.num|marital.status|       occupation|income|
+---+-------------+--------------+-----------------+------+
| 90|            9|       Widowed|                ?| <=50K|
| 82|            9|       Widowed|  Exec-managerial| <=50K|
| 66|           10|       Widowed|                ?| <=50K|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|
| 41|           10|     Separated|   Prof-specialty| <=50K|
+---+-------------+--------------+-----------------+------+
only showing top 5 rows
None


99

## **Handling missing data**

In [33]:
df = (df.withColumnRenamed("education.num", "education_num").withColumnRenamed("marital.status", "marital_status"))
df_cleaned = df.na.drop()
df_cleaned.count()

99

## **Handling missing data over spcific columns**

In [34]:
from pyspark.sql.functions import col

df_cleaned = df.where(col("education_num").isNotNull())

## **Replace nulls with specific values**

In [35]:
df_filled = df.na.fill({"education_num": 999})

## **Add new columns**

In [36]:
df = df.withColumn("age_plus_5", df["age"] + 5)
df.show(5)

+---+-------------+--------------+-----------------+------+----------+
|age|education_num|marital_status|       occupation|income|age_plus_5|
+---+-------------+--------------+-----------------+------+----------+
| 90|            9|       Widowed|                ?| <=50K|        95|
| 82|            9|       Widowed|  Exec-managerial| <=50K|        87|
| 66|           10|       Widowed|                ?| <=50K|        71|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|        59|
| 41|           10|     Separated|   Prof-specialty| <=50K|        46|
+---+-------------+--------------+-----------------+------+----------+
only showing top 5 rows


## **Rename columns**

In [37]:
df = df.withColumnRenamed("age", "years")
df.show(5)

+-----+-------------+--------------+-----------------+------+----------+
|years|education_num|marital_status|       occupation|income|age_plus_5|
+-----+-------------+--------------+-----------------+------+----------+
|   90|            9|       Widowed|                ?| <=50K|        95|
|   82|            9|       Widowed|  Exec-managerial| <=50K|        87|
|   66|           10|       Widowed|                ?| <=50K|        71|
|   54|            4|      Divorced|Machine-op-inspct| <=50K|        59|
|   41|           10|     Separated|   Prof-specialty| <=50K|        46|
+-----+-------------+--------------+-----------------+------+----------+
only showing top 5 rows


## **Drop columns**

In [38]:
df = df.drop("age_plus_5")
df.show(5)

+-----+-------------+--------------+-----------------+------+
|years|education_num|marital_status|       occupation|income|
+-----+-------------+--------------+-----------------+------+
|   90|            9|       Widowed|                ?| <=50K|
|   82|            9|       Widowed|  Exec-managerial| <=50K|
|   66|           10|       Widowed|                ?| <=50K|
|   54|            4|      Divorced|Machine-op-inspct| <=50K|
|   41|           10|     Separated|   Prof-specialty| <=50K|
+-----+-------------+--------------+-----------------+------+
only showing top 5 rows


## **Row operations**

- **`.filter()`**

In [42]:
filtered_df = df.filter(df['years'] > 50)
filtered_df.sort("years").show(10)

+-----+-------------+------------------+----------------+------+
|years|education_num|    marital_status|      occupation|income|
+-----+-------------+------------------+----------------+------+
|   51|           16|     Never-married|               ?|  >50K|
|   51|           10|Married-civ-spouse|Transport-moving| <=50K|
|   51|           10|          Divorced|    Adm-clerical|  >50K|
|   51|           13|          Divorced| Exec-managerial|  >50K|
|   51|           15|     Never-married|  Prof-specialty|  >50K|
|   51|           11|          Divorced|    Tech-support|  >50K|
|   51|           13|Married-civ-spouse|           Sales|  >50K|
|   52|           15|          Divorced| Exec-managerial|  >50K|
|   52|           13|           Widowed|   Other-service|  >50K|
|   53|            9|     Never-married|           Sales|  >50K|
+-----+-------------+------------------+----------------+------+
only showing top 10 rows


* **`.groupBy()`**

In [43]:
grouped_df = df.groupBy("marital_status").avg("education_num")
grouped_df.show()

+--------------------+------------------+
|      marital_status|avg(education_num)|
+--------------------+------------------+
|           Separated|            10.125|
|       Never-married|11.842105263157896|
|Married-spouse-ab...|              13.0|
|            Divorced|11.944444444444445|
|             Widowed| 9.333333333333334|
|  Married-civ-spouse| 11.97872340425532|
+--------------------+------------------+



จะใช้ **`.agg()`** ก็ได้

In [45]:
df.groupBy("marital_status").agg({"education_num": "avg"}).show()

+--------------------+------------------+
|      marital_status|avg(education_num)|
+--------------------+------------------+
|           Separated|            10.125|
|       Never-married|11.842105263157896|
|Married-spouse-ab...|              13.0|
|            Divorced|11.944444444444445|
|             Widowed| 9.333333333333334|
|  Married-civ-spouse| 11.97872340425532|
+--------------------+------------------+



# **Advanced DataFrame Operations**

- **`.join()`** to combine dataframes based on shared columns
- **`.union()`** to stack dataframes with the same schema
- arrays, maps, structs to handle nested and hierarchical data within DataFrames


## **Joins**

- **`df1.join(df2, on='colname', how='join_type')`**

In [ ]:
# Joining on column with the same name
df_joined = df1.join(df2, on='id', how='inner')

# Joining on columns with different names
df_joined = df1.join(df2, df1.id==df2.name, 'inner')

## **Union**

- **`df1.union(df2)`**
- Both dfs must have the same schema

# **Working with Arrays and Maps**

- Allow nested data within each row
- structured data within a single column

## **Arrays**

- Store lists within columns
   - **`ArrayType(StringType(), False)`**
       - **`StringType()`** = สมาชิกแต่ละตัวเป็น string
       - **`False`** = สมาชิกภายใน array ห้ามเป็น null
- For an array, define the values being passed using the appropriate datatype
- **`pyspark.sql.functions.lit()`** ย่อมาจาก literal ใช้สร้าง "ค่าคงที่" (constant value) ให้กลายเป็น Spark Column

In [ ]:
from pyspark.sql.functions import array, struct, lit

df = df.withColumn("scores", array(lit(85), lit(90), lit(78)))



```
+----+------------+
|name|scores      |
+----+------------+
|John|[85, 90, 78]|
|Mary|[85, 90, 78]|
|Bob |[85, 90, 78]|
+----+------------+
```



## **Maps**

- key-value pairs like dictionary
   - **`MapType(StringType(), StringType())`**
      - หมายถึง คอลัมน์นี้เก็บ Python dictionary ที่ key เป็น string และ value ก็เป็น string

In [48]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, MapType

schema = StructType([StructField('name', StringType(), True),
                     StructField('properties', MapType(StringType(), StringType()), True)])

## **Structs**

- Structs group related fields together within a single column
   - **`StructType(StructField, DataType())`**

In [ ]:
df = df.withColumn('full name', struct('first_name', 'last_name'))



```
+----------+---------+-------------+
|first_name|last_name|full_name    |
+----------+---------+-------------+
|John      |Smith    |{John, Smith}|
|Mary      |Jones    |{Mary, Jones}|
+----------+---------+-------------+
```



In [ ]:
from pyspark.sql.functions import array

df = df.withColumn("full_name", array("first_name", "last_name"))



```
+----------+---------+-------------+
|first_name|last_name|person       |
+----------+---------+-------------+
|John      |Smith    |[John, Smith]|
|Mary      |Jones    |[Mary, Jones]|
+----------+---------+-------------+
```



### ***Exercise: Joining flights with their destination airports***

- Examine the `airports` DataFrame. Note which key column will let you join airports to the `flights` table.



```
+----+--------------------+----------------+-----------------+----+---+---+
|dest|                name|             lat|              lon| alt| tz|dst|
+----+--------------------+----------------+-----------------+----+---+---+
| 04G|   Lansdowne Airport|      41.1304722|      -80.6195833|1044| -5|  A|
| 06A|Moton Field Munic...|      32.4605722|      -85.6800278| 264| -5|  A|
| 06C| Schaumburg Regional|      41.9893408|      -88.1012428| 801| -6|  A|
| 06N|     Randall Airport|       41.431912|      -74.3915611| 523| -5|  A|
| 09J|Jekyll Island Air...|      31.0744722|      -81.4277778|  11| -4|  A|
| 0A9|Elizabethton Muni...|      36.3712222|      -82.1734167|1593| -4|  A|
| 0G6|Williams County A...|      41.4673056|      -84.5067778| 730| -5|  A|
| 0G7|Finger Lakes Regi...|      42.8835647|      -76.7812318| 492| -5|  A|
| 0P2|Shoestring Aviati...|      39.7948244|      -76.6471914|1000| -5|  U|
| 0S9|Jefferson County ...|      48.0538086|     -122.8106436| 108| -8|  A|
| 0W3|Harford County Ai...|      39.5668378|      -76.2024028| 409| -5|  A|
| 10C|  Galt Field Airport|      42.4028889|      -88.3751111| 875| -6|  U|
| 17G|Port Bucyrus-Craw...|      40.7815556|      -82.9748056|1003| -5|  A|
| 19A|Jackson County Ai...|      34.1758638|      -83.5615972| 951| -4|  U|
| 1A3|Martin Campbell F...|      35.0158056|      -84.3468333|1789| -4|  A|
| 1B9| Mansfield Municipal|      42.0001331|      -71.1967714| 122| -5|  A|
| 1C9|Frazier Lake Airpark|54.0133333333333|-124.768333333333| 152| -8|  A|
| 1CS|Clow Internationa...|      41.6959744|      -88.1292306| 670| -6|  U|
| 1G3|  Kent State Airport|      41.1513889|      -81.4151111|1134| -4|  A|
| 1OH|     Fortman Airport|      40.5553253|      -84.3866186| 885| -5|  U|
+----+--------------------+----------------+-----------------+----+---+---+
```

- Join the `flights` with the `airports` DataFrame on the `"dest"` column. Save the result as `flights_with_airports`.



```
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute|
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|2014|   12|  8|     658|       -7|     935|       -5|     VX| N846VA|  1780|   SEA| LAX|     132|     954|   6|    58|
|2014|    1| 22|    1040|        5|    1505|        5|     AS| N559AS|   851|   SEA| HNL|     360|    2677|  10|    40|
|2014|    3|  9|    1443|       -2|    1652|        2|     VX| N847VA|   755|   SEA| SFO|     111|     679|  14|    43|
|2014|    4|  9|    1705|       45|    1839|       34|     WN| N360SW|   344|   PDX| SJC|      83|     569|  17|     5|
|2014|    3|  9|     754|       -1|    1015|        1|     AS| N612AS|   522|   SEA| BUR|     127|     937|   7|    54|
|2014|    1| 15|    1037|        7|    1352|        2|     WN| N646SW|    48|   PDX| DEN|     121|     991|  10|    37|
|2014|    7|  2|     847|       42|    1041|       51|     WN| N422WN|  1520|   PDX| OAK|      90|     543|   8|    47|
|2014|    5| 12|    1655|       -5|    1842|      -18|     VX| N361VA|   755|   SEA| SFO|      98|     679|  16|    55|
|2014|    4| 19|    1236|       -4|    1508|       -7|     AS| N309AS|   490|   SEA| SAN|     135|    1050|  12|    36|
|2014|   11| 19|    1812|       -3|    2352|       -4|     AS| N564AS|    26|   SEA| ORD|     198|    1721|  18|    12|
|2014|   11|  8|    1653|       -2|    1924|       -1|     AS| N323AS|   448|   SEA| LAX|     130|     954|  16|    53|
|2014|    8|  3|    1120|        0|    1415|        2|     AS| N305AS|   656|   SEA| PHX|     154|    1107|  11|    20|
|2014|   10| 30|     811|       21|    1038|       29|     AS| N433AS|   608|   SEA| LAS|     127|     867|   8|    11|
|2014|   11| 12|    2346|       -4|     217|      -28|     AS| N765AS|   121|   SEA| ANC|     183|    1448|  23|    46|
|2014|   10| 31|    1314|       89|    1544|      111|     AS| N713AS|   306|   SEA| SFO|     129|     679|  13|    14|
|2014|    1| 29|    2009|        3|    2159|        9|     UA| N27205|  1458|   PDX| SFO|      90|     550|  20|     9|
|2014|   12| 17|    2015|       50|    2150|       41|     AS| N626AS|   368|   SEA| SMF|      76|     605|  20|    15|
|2014|    8| 11|    1017|       -3|    1613|       -7|     WN| N8634A|   827|   SEA| MDW|     216|    1733|  10|    17|
|2014|    1| 13|    2156|       -9|     607|      -15|     AS| N597AS|    24|   SEA| BOS|     290|    2496|  21|    56|
|2014|    6|  5|    1733|      -12|    1945|      -10|     OO| N215AG|  3488|   PDX| BUR|     111|     817|  17|    33|
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
```



In [ ]:
flights_with_airports = flights.join(airports, on='dest', how='leftouter')



```
 +----+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+--------+--------+----+------+--------------------+---------+-----------+----+---+---+
    |dest|year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|air_time|distance|hour|minute|                name|      lat|        lon| alt| tz|dst|
    +----+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+--------+--------+----+------+--------------------+---------+-----------+----+---+---+
    | LAX|2014|   12|  8|     658|       -7|     935|       -5|     VX| N846VA|  1780|   SEA|     132|     954|   6|    58|    Los Angeles Intl|33.942536|-118.408075| 126| -8|  A|
    | HNL|2014|    1| 22|    1040|        5|    1505|        5|     AS| N559AS|   851|   SEA|     360|    2677|  10|    40|       Honolulu Intl|21.318681|-157.922428|  13|-10|  N|
    | SFO|2014|    3|  9|    1443|       -2|    1652|        2|     VX| N847VA|   755|   SEA|     111|     679|  14|    43|  San Francisco Intl|37.618972|-122.374889|  13| -8|  A|
    | SJC|2014|    4|  9|    1705|       45|    1839|       34|     WN| N360SW|   344|   PDX|      83|     569|  17|     5|Norman Y Mineta S...|  37.3626|-121.929022|  62| -8|  A|
    | BUR|2014|    3|  9|     754|       -1|    1015|        1|     AS| N612AS|   522|   SEA|     127|     937|   7|    54|            Bob Hope|34.200667|-118.358667| 778| -8|  A|
    | DEN|2014|    1| 15|    1037|        7|    1352|        2|     WN| N646SW|    48|   PDX|     121|     991|  10|    37|         Denver Intl|39.861656|-104.673178|5431| -7|  A|
    | OAK|2014|    7|  2|     847|       42|    1041|       51|     WN| N422WN|  1520|   PDX|      90|     543|   8|    47|Metropolitan Oakl...|37.721278|-122.220722|   9| -8|  A|
    | SFO|2014|    5| 12|    1655|       -5|    1842|      -18|     VX| N361VA|   755|   SEA|      98|     679|  16|    55|  San Francisco Intl|37.618972|-122.374889|  13| -8|  A|
    | SAN|2014|    4| 19|    1236|       -4|    1508|       -7|     AS| N309AS|   490|   SEA|     135|    1050|  12|    36|      San Diego Intl|32.733556|-117.189667|  17| -8|  A|
    | ORD|2014|   11| 19|    1812|       -3|    2352|       -4|     AS| N564AS|    26|   SEA|     198|    1721|  18|    12|  Chicago Ohare Intl|41.978603| -87.904842| 668| -6|  A|
    | LAX|2014|   11|  8|    1653|       -2|    1924|       -1|     AS| N323AS|   448|   SEA|     130|     954|  16|    53|    Los Angeles Intl|33.942536|-118.408075| 126| -8|  A|
    | PHX|2014|    8|  3|    1120|        0|    1415|        2|     AS| N305AS|   656|   SEA|     154|    1107|  11|    20|Phoenix Sky Harbo...|33.434278|-112.011583|1135| -7|  N|
    | LAS|2014|   10| 30|     811|       21|    1038|       29|     AS| N433AS|   608|   SEA|     127|     867|   8|    11|      Mc Carran Intl|36.080056| -115.15225|2141| -8|  A|
    | ANC|2014|   11| 12|    2346|       -4|     217|      -28|     AS| N765AS|   121|   SEA|     183|    1448|  23|    46|Ted Stevens Ancho...|61.174361|-149.996361| 152| -9|  A|
    | SFO|2014|   10| 31|    1314|       89|    1544|      111|     AS| N713AS|   306|   SEA|     129|     679|  13|    14|  San Francisco Intl|37.618972|-122.374889|  13| -8|  A|
    | SFO|2014|    1| 29|    2009|        3|    2159|        9|     UA| N27205|  1458|   PDX|      90|     550|  20|     9|  San Francisco Intl|37.618972|-122.374889|  13| -8|  A|
    | SMF|2014|   12| 17|    2015|       50|    2150|       41|     AS| N626AS|   368|   SEA|      76|     605|  20|    15|     Sacramento Intl|38.695417|-121.590778|  27| -8|  A|
    | MDW|2014|    8| 11|    1017|       -3|    1613|       -7|     WN| N8634A|   827|   SEA|     216|    1733|  10|    17| Chicago Midway Intl|41.785972| -87.752417| 620| -6|  A|
    | BOS|2014|    1| 13|    2156|       -9|     607|      -15|     AS| N597AS|    24|   SEA|     290|    2496|  21|    56|General Edward La...|42.364347| -71.005181|  19| -5|  A|
    | BUR|2014|    6|  5|    1733|      -12|    1945|      -10|     OO| N215AG|  3488|   PDX|     111|     817|  17|    33|            Bob Hope|34.200667|-118.358667| 778| -8|  A|
    +----+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+--------+--------+----+------+--------------------+---------+-----------+----+---+---+
```



# **User-Defined Functions**

- Use PySpark UDFs for simple, small-scale tasks, and pandas UDFs for larger, more complex data processing.

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MySparkApp").getOrCreate()
schema = StructType([StructField("age",IntegerType()),
                     StructField("education.num", IntegerType()),
                     StructField("marital.status",StringType()),
                     StructField("occupation", StringType()),
                     StructField("income", StringType()),
                    ])
df = spark.read.json("adults.json", schema=schema)
df = df.withColumnRenamed("education.num", "education_num")
df = df.withColumnRenamed("marital.status", "marital_status")
df.show(5)

+---+-------------+--------------+-----------------+------+
|age|education_num|marital_status|       occupation|income|
+---+-------------+--------------+-----------------+------+
| 90|            9|       Widowed|                ?| <=50K|
| 82|            9|       Widowed|  Exec-managerial| <=50K|
| 66|           10|       Widowed|                ?| <=50K|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|
| 41|           10|     Separated|   Prof-specialty| <=50K|
+---+-------------+--------------+-----------------+------+
only showing top 5 rows


## **PySpark UDF**

In [12]:
from pyspark.sql.functions import udf, pandas_udf

# Define function
def to_uppercase(s):
    return s.upper() if s else None

# Register function
to_uppercase_udf = udf(to_uppercase, StringType())

df = df.withColumn("marital_status_upper", to_uppercase_udf(df['marital_status']))
df.show(5)

+---+-------------+--------------+-----------------+------+--------------------+
|age|education_num|marital_status|       occupation|income|marital_status_upper|
+---+-------------+--------------+-----------------+------+--------------------+
| 90|            9|       Widowed|                ?| <=50K|             WIDOWED|
| 82|            9|       Widowed|  Exec-managerial| <=50K|             WIDOWED|
| 66|           10|       Widowed|                ?| <=50K|             WIDOWED|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|            DIVORCED|
| 41|           10|     Separated|   Prof-specialty| <=50K|           SEPARATED|
+---+-------------+--------------+-----------------+------+--------------------+
only showing top 5 rows


## **Pandas UDF**

In [13]:
@pandas_udf("float")
def age_month_udf(age):
    return age * 12

df = df.withColumn("age_month", age_month_udf(df['age']))
df.show(5)

+---+-------------+--------------+-----------------+------+--------------------+---------+
|age|education_num|marital_status|       occupation|income|marital_status_upper|age_month|
+---+-------------+--------------+-----------------+------+--------------------+---------+
| 90|            9|       Widowed|                ?| <=50K|             WIDOWED|   1080.0|
| 82|            9|       Widowed|  Exec-managerial| <=50K|             WIDOWED|    984.0|
| 66|           10|       Widowed|                ?| <=50K|             WIDOWED|    792.0|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|            DIVORCED|    648.0|
| 41|           10|     Separated|   Prof-specialty| <=50K|           SEPARATED|    492.0|
+---+-------------+--------------+-----------------+------+--------------------+---------+
only showing top 5 rows


### ***Exercise: Integers in PySpark UDFs***

- Register the function `age_category` as a UDF called `age_category_udf`.
- Add a new column to the DataFrame `df` called `"category"` that applies the UDF to categorize people based on their age. The argument for `age_category_udf()` is provided for you.

In [14]:
def age_category(age):
    if age < 18:
        return 'Child'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

# Register the function age_category as a UDF
age_category_udf = udf(age_category, StringType())

df = df.withColumn("category", age_category_udf(df["age"]))
df.show(5)

+---+-------------+--------------+-----------------+------+--------------------+---------+--------+
|age|education_num|marital_status|       occupation|income|marital_status_upper|age_month|category|
+---+-------------+--------------+-----------------+------+--------------------+---------+--------+
| 90|            9|       Widowed|                ?| <=50K|             WIDOWED|   1080.0|  Senior|
| 82|            9|       Widowed|  Exec-managerial| <=50K|             WIDOWED|    984.0|  Senior|
| 66|           10|       Widowed|                ?| <=50K|             WIDOWED|    792.0|  Senior|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|            DIVORCED|    648.0|   Adult|
| 41|           10|     Separated|   Prof-specialty| <=50K|           SEPARATED|    492.0|   Adult|
+---+-------------+--------------+-----------------+------+--------------------+---------+--------+
only showing top 5 rows


### ***Exercise: Pandas UDF***

In [18]:
from pyspark.sql.types import DoubleType

# Define a Pandas UDF that adds 10 to each element in a vectorized way
@pandas_udf(DoubleType())
def add_ten_pandas(column):
    return column + 10

# Apply the UDF and show the result
df = df.withColumn("10_plus_edu", add_ten_pandas(df['education_num']))
df.show(5)

+---+-------------+--------------+-----------------+------+--------------------+---------+--------+-----------+
|age|education_num|marital_status|       occupation|income|marital_status_upper|age_month|category|10_plus_edu|
+---+-------------+--------------+-----------------+------+--------------------+---------+--------+-----------+
| 90|            9|       Widowed|                ?| <=50K|             WIDOWED|   1080.0|  Senior|       19.0|
| 82|            9|       Widowed|  Exec-managerial| <=50K|             WIDOWED|    984.0|  Senior|       19.0|
| 66|           10|       Widowed|                ?| <=50K|             WIDOWED|    792.0|  Senior|       20.0|
| 54|            4|      Divorced|Machine-op-inspct| <=50K|            DIVORCED|    648.0|   Adult|       14.0|
| 41|           10|     Separated|   Prof-specialty| <=50K|           SEPARATED|    492.0|   Adult|       20.0|
+---+-------------+--------------+-----------------+------+--------------------+---------+--------+-----

# **Resilient Distributed Datasets**

- Immutable and can be transformed using operations like **`map()`** or **`filter()`**
    - **`.map()`** applies functions: **`rdd.map(func)`**
- **`collect()`** or **`pararellize()`** retrieve results or create RDDs
    - **`.collect()`** collects data across clusters
- Data are distriuted across clusters when RDDs are created

In [20]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RDDExample").getOrCreate()
schema = StructType([StructField("age",IntegerType()),
                     StructField("education.num", IntegerType()),
                     StructField("marital.status",StringType()),
                     StructField("occupation", StringType()),
                     StructField("income", StringType()),
                    ])
df = spark.read.json("adults.json", schema=schema)
df = df.withColumnRenamed("education.num", "education_num")
df = df.withColumnRenamed("marital.status", "marital_status")

# Convert df to RDD
rdd = df.rdd

# Show RDD
rdd.collect()

[Row(age=90, education_num=9, marital_status='Widowed', occupation='?', income='<=50K'),
 Row(age=82, education_num=9, marital_status='Widowed', occupation='Exec-managerial', income='<=50K'),
 Row(age=66, education_num=10, marital_status='Widowed', occupation='?', income='<=50K'),
 Row(age=54, education_num=4, marital_status='Divorced', occupation='Machine-op-inspct', income='<=50K'),
 Row(age=41, education_num=10, marital_status='Separated', occupation='Prof-specialty', income='<=50K'),
 Row(age=34, education_num=9, marital_status='Divorced', occupation='Other-service', income='<=50K'),
 Row(age=38, education_num=6, marital_status='Separated', occupation='Adm-clerical', income='<=50K'),
 Row(age=74, education_num=16, marital_status='Never-married', occupation='Prof-specialty', income='>50K'),
 Row(age=68, education_num=9, marital_status='Divorced', occupation='Prof-specialty', income='<=50K'),
 Row(age=41, education_num=10, marital_status='Never-married', occupation='Craft-repair', in

In [22]:
data_collected = df.collect()
for row in data_collected:
    print(row)

Row(age=90, education_num=9, marital_status='Widowed', occupation='?', income='<=50K')
Row(age=82, education_num=9, marital_status='Widowed', occupation='Exec-managerial', income='<=50K')
Row(age=66, education_num=10, marital_status='Widowed', occupation='?', income='<=50K')
Row(age=54, education_num=4, marital_status='Divorced', occupation='Machine-op-inspct', income='<=50K')
Row(age=41, education_num=10, marital_status='Separated', occupation='Prof-specialty', income='<=50K')
Row(age=34, education_num=9, marital_status='Divorced', occupation='Other-service', income='<=50K')
Row(age=38, education_num=6, marital_status='Separated', occupation='Adm-clerical', income='<=50K')
Row(age=74, education_num=16, marital_status='Never-married', occupation='Prof-specialty', income='>50K')
Row(age=68, education_num=9, marital_status='Divorced', occupation='Prof-specialty', income='<=50K')
Row(age=41, education_num=10, marital_status='Never-married', occupation='Craft-repair', income='>50K')
Row(ag

# **Spark SQL**

In [23]:
spark = SparkSession.builder.appName("Spark SQL").getOrCreate()
data = [("Alice", "HR", 30), ("Bob", "IT", 40), ("Cathy", "HR", 28)]
columns = ["Name", "Department", "Age"]

df = spark.createDataFrame(data, schema=columns)
df.show()

+-----+----------+---+
| Name|Department|Age|
+-----+----------+---+
|Alice|        HR| 30|
|  Bob|        IT| 40|
|Cathy|        HR| 28|
+-----+----------+---+



In [26]:
# Register DataFrame as a temporary view
df.createOrReplaceTempView("people")
result = spark.sql('''SELECT Name, Age
                      FROM people
                      WHERE age > 30''')
result.show()

+----+---+
|Name|Age|
+----+---+
| Bob| 40|
+----+---+



In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark SQL").getOrCreate()
df = spark.read.csv('salaries.csv', header=True, inferSchema=True)

# Register dataframe as a temporary view
df.createOrReplaceTempView("employees")

# SQL query result
query = spark.sql('''SELECT experience_level, job_title, salary
                     FROM employees
                     WHERE salary > 500000''')
query.show(5)

+----------------+--------------+--------+
|experience_level|     job_title|  salary|
+----------------+--------------+--------+
|              SE| Data Engineer|  720000|
|              MI|Data Scientist|11000000|
|              MI|Data Scientist| 3000000|
|              EN| Data Engineer| 4450000|
|              SE|Data Scientist| 4000000|
+----------------+--------------+--------+
only showing top 5 rows


In [12]:
# DataFrame transformation

high_income = query.withColumn("Bonus", query['salary'] * 0.1)
high_income.show(5)

+----------------+--------------+--------+---------+
|experience_level|     job_title|  salary|    Bonus|
+----------------+--------------+--------+---------+
|              SE| Data Engineer|  720000|  72000.0|
|              MI|Data Scientist|11000000|1100000.0|
|              MI|Data Scientist| 3000000| 300000.0|
|              EN| Data Engineer| 4450000| 445000.0|
|              SE|Data Scientist| 4000000| 400000.0|
+----------------+--------------+--------+---------+
only showing top 5 rows


In [24]:
# More example
result = spark.sql("""SELECT job_title, AVG(salary) AS Average_Salary
                      FROM employees
                      GROUP BY job_title
                      ORDER BY Average_Salary DESC""")
result.show(5, truncate=False)

+--------------------------------+------------------+
|job_title                       |Average_Salary    |
+--------------------------------+------------------+
|Principal Data Architect        |3000000.0         |
|AI Software Development Engineer|2100000.0         |
|Lead Machine Learning Engineer  |1566200.0         |
|Manager Data Management         |1562500.0         |
|Lead Data Analyst               |1095833.3333333333|
+--------------------------------+------------------+
only showing top 5 rows


In [25]:
df.groupBy('job_title').avg('salary').sort('avg(salary)', ascending=False).show(5, truncate=False)

+--------------------------------+------------------+
|job_title                       |avg(salary)       |
+--------------------------------+------------------+
|Principal Data Architect        |3000000.0         |
|AI Software Development Engineer|2100000.0         |
|Lead Machine Learning Engineer  |1566200.0         |
|Manager Data Management         |1562500.0         |
|Lead Data Analyst               |1095833.3333333333|
+--------------------------------+------------------+
only showing top 5 rows


In [27]:
# More example
result = spark.sql('''SELECT job_title, salary_in_usd
                      FROM employees
                      WHERE company_location == "CA"''')
result.describe().show()

+-------+--------------------+-----------------+
|summary|           job_title|    salary_in_usd|
+-------+--------------------+-----------------+
|  count|                1115|             1115|
|   mean|                NULL|143228.7273542601|
| stddev|                NULL|61839.75996920976|
|    min|        AI Architect|            15000|
|    max|Statistical Progr...|           800000|
+-------+--------------------+-----------------+



In [3]:
df.show(1)

+---------+----------------+---------------+-------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|          job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+-------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|     2020|              EN|             FT|Azure Data Engineer|100000|            USD|       100000|                MU|           0|              MU|           S|
+---------+----------------+---------------+-------------------+------+---------------+-------------+------------------+------------+----------------+------------+
only showing top 1 row


# **PySpark SQL Aggregation**

In [5]:
spark.sql('''SELECT experience_level, SUM(salary_in_usd) AS Total_Salary, AVG(salary_in_usd) AS Average_Salary
             FROM employees
             GROUP BY experience_level
             ORDER BY Average_Salary DESC''').show()

+----------------+------------+------------------+
|experience_level|Total_Salary|    Average_Salary|
+----------------+------------+------------------+
|              EX|   162927258|198208.34306569342|
|              SE|  3928773856|174433.86120854237|
|              MI|  1546123981|144187.63228574092|
|              EN|   339743721|107310.08243840809|
+----------------+------------+------------------+



## **Combining DataFrame and SQL Operations**

In [13]:
filtered_df = df.filter(df['salary_in_usd'] > 300000)

# Register filtered dataframe as a view
filtered_df.createOrReplaceTempView("filtered_employees")

spark.sql('''SELECT experience_level, COUNT(*) AS Employee_Count
             FROM filtered_employees
             GROUP BY experience_level
             ORDER BY Employee_Count DESC''').show()

+----------------+--------------+
|experience_level|Employee_Count|
+----------------+--------------+
|              SE|          1037|
|              MI|           265|
|              EX|            69|
|              EN|            29|
+----------------+--------------+



In [21]:
# Example of type casting
data = [("HR", "3000"), ("IT", "4000"), ("Finance", "3500")]
columns = ["Department", "Salary"]
df = spark.createDataFrame(data, schema=columns)
df.show()

+----------+------+
|Department|Salary|
+----------+------+
|        HR|  3000|
|        IT|  4000|
|   Finance|  3500|
+----------+------+



In [19]:
df.groupBy('Department').sum('Salary').show()

AnalysisException: "Salary" is not a numeric column. Aggregation function can only be applied on a numeric column.

In [20]:
df = df.withColumn('Salary', df['Salary'].cast('int'))
df.groupBy('Department').sum('Salary').show()

+----------+-----------+
|Department|sum(Salary)|
+----------+-----------+
|        HR|       3000|
|   Finance|       3500|
|        IT|       4000|
+----------+-----------+



# **RDDs for aggregations**

- You must use lambda functions

In [22]:
data = [("HR", "3000"), ("IT", "4000"), ("Finance", "3500")]
columns = ["Department", "Salary"]
df = spark.createDataFrame(data, schema=columns)
df.show()

+----------+------+
|Department|Salary|
+----------+------+
|        HR|  3000|
|        IT|  4000|
|   Finance|  3500|
+----------+------+



In [25]:
# สร้าง Pair RDD ซึ่งเป็นรูปแบบที่ reduceByKey() ต้องการ
rdd = df.rdd.map(lambda row: (row['Department'], row['Salary']))
# ('HR', '3000')
# ('IT', '4000')
# ('Finance', '3500')

# รวมค่าที่มี key เดียวกันเข้าด้วยกัน
rdd_aggregated = rdd.reduceByKey(lambda x, y: x + y)
print(rdd_aggregated.collect())

[('HR', '3000'), ('IT', '4000'), ('Finance', '3500')]


Better examples


In [28]:
data = [("HR", 3000), ("HR", 2000), ("IT", 4000), ("IT", 1000), ("Finance", 3500)]
columns = ["Department", "Salary"]
df = spark.createDataFrame(data, schema=columns)
df.show()

+----------+------+
|Department|Salary|
+----------+------+
|        HR|  3000|
|        HR|  2000|
|        IT|  4000|
|        IT|  1000|
|   Finance|  3500|
+----------+------+



In [34]:
df.rdd.collect()

[Row(Department='HR', Salary=3000),
 Row(Department='HR', Salary=2000),
 Row(Department='IT', Salary=4000),
 Row(Department='IT', Salary=1000),
 Row(Department='Finance', Salary=3500)]

In [35]:
rdd = df.rdd.map(lambda row: (row['Department'], row['Salary']))
print(rdd.collect())

[('HR', 3000), ('HR', 2000), ('IT', 4000), ('IT', 1000), ('Finance', 3500)]


In [36]:
rdd_aggregated = rdd.reduceByKey(lambda x, y: x + y)
print(rdd_aggregated.collect())

[('HR', 5000), ('IT', 5000), ('Finance', 3500)]


### **Exercise: Aggregating in PySpark**

#### **Instructions**

- Find the minimum salary at a US, Small company - performing the filtering by referencing the column directly (`"salary_in_usd"`), not passing an SQL string.
- Find the maximum salary at a US, Large company, denoted by a `"L"` - performing the filtering by referencing the column directly (`"salary_in_usd"`), not passing a SQL string.

In [37]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark SQL").getOrCreate()
salaries_df = spark.read.csv('salaries.csv', header=True, inferSchema=True)

# Find the minimum salaries for small companies
salaries_df.filter(salaries_df.company_size == "S").groupBy().min("salary_in_usd").show()

# Find the maximum salaries for large companies
salaries_df.filter(salaries_df.company_size=='L').groupBy().max("salary_in_usd").show()

+------------------+
|min(salary_in_usd)|
+------------------+
|             15809|
+------------------+

+------------------+
|max(salary_in_usd)|
+------------------+
|            423000|
+------------------+



- Calculate the average salaries of large US companies using the `"salary_in_usd"` column.
- Calculate the total salaries of large US companies.

In [38]:
# Average salaries at large us companies
large_companies=salaries_df.filter(salaries_df.company_size == "L").filter(salaries_df.company_location == "US").groupBy().avg("salary_in_usd")

#set a large companies variable for other analytics
large_companies=salaries_df.filter(salaries_df.company_size == "L").filter(salaries_df.company_location == "US")

# Total salaries in usd
large_companies.groupBy().sum("salary_in_usd").show()

+------------------+
|sum(salary_in_usd)|
+------------------+
|         194256875|
+------------------+



# **Scaling to Large Data**

- **`.explain()`** describes Spark's execution plans and identify bottlenecks
- **`.cache()`** and **`.persist()`** reuse intermediate results and speed up workflows
- **`.unpersist()`** frees resources when they are no longer used
- Best practices
  - Avoiding unnecessary shuffles
  - Use broadcast joins for small datasets
  - Minimize repeated actions



## **`pyspark.sql.functions.broadcast()`**

 The **`.broadcast()`** method is used to distribute a small dataset across all worker nodes, minimizing shuffling during join operations.

In [ ]:
from pyspark.sql.functions import broadcast

joined_df = large_df.join(broadcast(small_df), on='key_column', how='inner')

## **`.explain()`**

- Explain execution plans

In [4]:
df = spark.read.csv('salaries.csv', header=True, inferSchema=True)
df.filter(df['experience_level']=='EN').select(['experience_level', 'job_title']).show(5)

+----------------+-------------------+
|experience_level|          job_title|
+----------------+-------------------+
|              EN|Azure Data Engineer|
|              EN| Staff Data Analyst|
|              EN|       Data Analyst|
|              EN|       Data Analyst|
|              EN|     Data Scientist|
+----------------+-------------------+
only showing top 5 rows


In [5]:
df.filter(df['experience_level']=='EN').select(['experience_level', 'job_title']).explain()

== Physical Plan ==
*(1) Filter (isnotnull(experience_level#127) AND (experience_level#127 = EN))
+- FileScan csv [experience_level#127,job_title#129] Batched: false, DataFilters: [isnotnull(experience_level#127), (experience_level#127 = EN)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/salaries.csv], PartitionFilters: [], PushedFilters: [IsNotNull(experience_level), EqualTo(experience_level,EN)], ReadSchema: struct<experience_level:string,job_title:string>




## **`.cache()` and `.persist()`**

In [6]:
# Subsequent operations reuse "cache" version
df.cache()

df.filter(df['experience_level']=='EN').show(5)
df.groupBy('experience_level').count().show(5)

+---------+----------------+---------------+-------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|          job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+-------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|     2020|              EN|             FT|Azure Data Engineer|100000|            USD|       100000|                MU|           0|              MU|           S|
|     2020|              EN|             CT| Staff Data Analyst| 60000|            CAD|        44753|                CA|          50|              CA|           L|
|     2020|              EN|             FT|       Data Analyst| 42000|            EUR|        47899|                DE|           0|              DE|           L|
|     2020|     

## **Persisting dataframes with different storage levels**

- Useful for long-running jobs on large clusters

In [8]:
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)

result = df.groupBy('experience_level').agg({'salary_in_usd':'avg'}).show(5)

+----------------+------------------+
|experience_level|avg(salary_in_usd)|
+----------------+------------------+
|              EX|198208.34306569342|
|              MI|144187.63228574092|
|              EN|107310.08243840809|
|              SE|174433.86120854237|
+----------------+------------------+



## **`.unpersist()` to free up resources**

In [9]:
df.unpersist()

DataFrame[work_year: int, experience_level: string, employment_type: string, job_title: string, salary: int, salary_currency: string, salary_in_usd: int, employee_residence: string, remote_ratio: int, company_location: string, company_size: string]

### **Exercise: Bringing it all together I**
- Import **`SparkSession`** from **`pyspark.sql`**.
- Make a new **`SparkSession`** called `final_spark` using **`SparkSession.builder.getOrCreate()`**.
- Print `my_spark` to the console to verify it's a **`SparkSession`**.
- Create a new DataFrame from a preloaded schema and column definition.

In [12]:
data = [('Alice', 'HR', 3000), ('Bob', 'IT', 4000), ('Cathy', 'HR', 3500)]
columns = ['Name', 'Department', 'Salary']

# Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

# Create my_spark
my_spark = SparkSession.builder.appName("final_spark").getOrCreate()

# Print my_spark
print(my_spark)

# Load dataset into a DataFrame
df = my_spark.createDataFrame(data, schema=columns)

df.show()

+-----+----------+------+
| Name|Department|Salary|
+-----+----------+------+
|Alice|        HR|  3000|
|  Bob|        IT|  4000|
|Cathy|        HR|  3500|
+-----+----------+------+



### **Exercise: Bringing it all together II**

##### **Instructions**

- Cache the `df` DataFrame.
- Explain the processing of the `agg_result` DataFrame.
- Unpersist the cached `df` DataFrame after processing.

In [13]:
# Cache the DataFrame
df.cache()

# Perform aggregation
agg_result = df.groupBy("Department").sum("Salary")
agg_result.show()

# Analyze the execution plan
agg_result.explain()

# Uncache the DataFrame
df.unpersist()

+----------+-----------+
|Department|sum(Salary)|
+----------+-----------+
|        HR|       6500|
|        IT|       4000|
+----------+-----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Department#1115], functions=[sum(Salary#1116L)])
   +- Exchange hashpartitioning(Department#1115, 200), ENSURE_REQUIREMENTS, [plan_id=345]
      +- HashAggregate(keys=[Department#1115], functions=[partial_sum(Salary#1116L)])
         +- InMemoryTableScan [Department#1115, Salary#1116L]
               +- InMemoryRelation [Name#1114, Department#1115, Salary#1116L], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- *(1) Scan ExistingRDD[Name#1114,Department#1115,Salary#1116L]




DataFrame[Name: string, Department: string, Salary: bigint]

We cached the DataFrame `df` for speed, conducted analytics with `agg_result` for seperation, seeing how it executes, and now we can unpersist the original `df`. We don't need to unpersist `agg_result`, as it was never cached.